In [1]:
import boto3
import botocore
from botocore.config import Config
from botocore.exceptions import ClientError
from boto3.dynamodb.conditions import Key

import requests
import json


region_name = 'us-east-1'
my_files_table_name = "algolab-askyourpdf-dev-my_files",
my_file_page_content_table_name = "algolab-askyourpdf-dev-my_file_page_content"

In [2]:
def get_secret_key(key_name='smtp2go-key'):

    secret_name = "sean-gpt-service-key"
    
    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    secret = json.loads(get_secret_value_response['SecretString'])

    # Your code goes here.
    return secret[key_name]

In [3]:
def send_email(recipient_email, subject, body):
    
    try:

        url = 'https://api.smtp2go.com/v3/email/send'
        data = {
          "api_key": get_secret_key('smtp2go-key'),
          "to": [
            f"{recipient_email} <{recipient_email}>"
          ],
          "sender": "AskYourPdf 팀 <sean@52g.team>",
          "subject": subject,
          "text_body": body,
          "custom_headers": [
            {
              "header": "Reply-To",
              "value": "AskYourPdf 팀 <sean@52g.team>"
            }
          ]
        }
        json_data = json.dumps(data)


        # Content-Type을 application/json으로 설정
        headers = {'Content-Type': 'application/json'}

        response = requests.post(url, data=json_data, headers=headers)

        result = json.loads(response.text)
        
        if result['data']['succeeded'] == 1:
            return True
        else:
            return False
    except Exception as e:
        print(e)
        return False        

In [4]:
def update_record_to_my_files(record, my_files_table_name):
    """
    지정된 테이블에 항목 넣기

    args:
        my_files_table_name (str): 테이블 이름
        record (dict): 넣을 항목
    """
    try:
        # dynamodb 리소스 생성
        dynamodb = boto3.resource('dynamodb', region_name=region_name)

        # 테이블 객체 생성
        table = dynamodb.Table(my_files_table_name)

        # 항목 넣기
        response = table.put_item(
            Item=record
        )
        # print(response)
        # 응답 반환
        return response
    except Exception as e:
        raise e
        
        
def get_record_my_file_page_content(username, file_hashkey, page_no, my_file_page_content_table_name):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    table = dynamodb.Table(my_file_page_content_table_name)

    try:
        response = table.get_item(
            Key={
                'hashkey': username+'-'+file_hashkey,
                'page_no': page_no
            }
        )
        item = response.get('Item')
        return item
    except Exception as e:
        print("Failed to get record:", e)
        return None   
    
    
def get_my_file_by_keys(username, file_hashkey, my_files_table_name):
    """
    Get a record from the DynamoDB table 'my_files' using username and file_hashkey.

    Args:
    - username (str): Username associated with the record.
    - file_hashkey (str): Hash key of the record.

    Returns:
    - dict: Record data if found, None otherwise.
    """
    dynamodb = boto3.resource('dynamodb', region_name=region_name)

    try:
        table = dynamodb.Table(my_files_table_name)
        response = table.query(
                    KeyConditionExpression=(
                        Key('username').eq(username) &
                        Key('file_hashkey').eq(file_hashkey)
                    )
                )        
        
        item = response['Items'][0]
        return item
    except Exception as e:
        print("Failed to get record:", e)
        return None    

In [5]:
def query_dynamodb_table(table_name, key, page_no):
    """
    DynamoDB 테이블에서 특정 조건에 맞는 레코드를 가져오는 함수

    :param table_name: DynamoDB 테이블 이름
    :param partition_key: Partition Key 값
    :param sort_key_condition: Sort Key 조건
    :return: 조건에 맞는 레코드 리스트
    """
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(table_name)

    try:
        response = table.query(
            KeyConditionExpression=Key('hashkey').eq(key) & Key('page_no').lte(page_no)
        )
        return response.get('Items', [])
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

In [6]:
def main(username, hash_key, my_files_table_name, my_file_page_content_table_name):
    
    # get record
    try:
        record = get_my_file_by_keys(username, hash_key, my_files_table_name)
        key = username+'-'+hash_key
        snippets = query_dynamodb_table(my_file_page_content_table_name,key,-100)
        # record 가 없을때
        if record is None or len(snippets) == 0:
            msg = '문서 해석 완료 알림에 문제가 생겼어요'
            return {
                'statusCode': 500,
                'username': username,
                'hash_key': hash_key,
                'my_files_table_name': my_files_table_name,
                'my_file_page_content_table_name': my_file_page_content_table_name,
                'error': msg
            }
        snippet_summary = ''
        sorted_data = sorted(snippets, key=lambda x: x['page_no'], reverse=True)
        for item in sorted_data:
            snippet_summary = snippet_summary+item['llm_content']+'\n'
        record['page_per_summary'] = snippet_summary
        update_record_to_my_files(record, my_files_table_name)
        recipient_email = username
        subject = 'AskYourPdf [{}] 해석이 완료되었습니다'.format(record['file_name'])
        body = '''
안녕하세요!

AskYourPdf 에서 알려 드립니다.

요청하신 [{}] 파일이 지식화 되어, 언제든지 묻고 답하실 수 있습니다.

https://askyourpdf.algolab.team

감사합니다
AskYourPdf 팀
'''.format(record['file_name'])
        send_email(recipient_email, subject, body)
        
        
        return {
            'file_name': record['file_name'],
            'username': username,
            'hash_key': hash_key,
            'msg': '파일의 읽기, 지식화, 요약, 페이지별 번역/요약이 완료되었습니다'
        }
    except Exception as e:
        print(e)
        msg = '문서 해석 완료 알림에 문제가 생겼어요: ' + str(e)
        return {
            'statusCode': 500,
            'username': username,
            'hash_key': hash_key,
            'my_files_table_name': my_files_table_name,
            'my_file_page_content_table_name': my_file_page_content_table_name,
            'error': msg
        }
        

In [7]:
def lambda_handler(event, context):
    try:
        username = ''
        hash_key = ''
        my_files_table_name = ''
        my_file_page_content_table_name = ''

        for item in event:
            if 'username' in item:
                username = item['username']
            if 'hash_key' in item:
                hash_key = item['hash_key']
            if 'my_files_table_name' in item:
                my_files_table_name = item['my_files_table_name']
            if 'my_file_page_content_table_name' in item:
                my_file_page_content_table_name = item['my_file_page_content_table_name']
        
        if (username=='' or hash_key=='' or my_files_table_name==''):
            return {
                'statusCode': 500,
                'error': 'validation failed',
                'username': username,
                'hash_key': hash_key,
                'my_files_table_name': my_files_table_name,
                'my_file_page_content_table_name': my_file_page_content_table_name,
            }             
        res = main(username, hash_key, my_files_table_name, my_file_page_content_table_name)

        return res
    except Exception as e:
        print(e)
        msg = '문서 해석 완료 알림에 문제가 생겼어요: ' + str(e)
        return {
            'statusCode': 500,
            'username': username,
            'hash_key': hash_key,
            'my_files_table_name': my_files_table_name,
            'my_file_page_content_table_name': my_file_page_content_table_name,
            'error': msg
        }

In [8]:
event = [
  {
    "statusCode": 200,
    "username": "hmkim@gsenergy.co.kr",
    "hash_key": "083263112afe6caa",
    "my_files_table_name": "algolab-askyourpdf-dev-my_files",
    "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
  },
  [
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "index_no": 0,
      "page_no": -100,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    }
  ],
  [
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 1,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 2,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 3,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 4,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 5,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 6,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 7,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 8,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 9,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 10,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 11,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 12,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 13,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 14,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 15,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 16,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 17,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 18,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 19,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 20,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 21,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 22,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 23,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 24,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 25,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 26,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 27,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 28,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 29,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 30,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 31,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 32,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 33,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 34,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 35,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 36,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 37,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 38,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 39,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 40,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 41,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 42,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 43,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 44,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 45,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 46,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 47,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 48,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 49,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 50,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    },
    {
      "statusCode": 200,
      "username": "hmkim@gsenergy.co.kr",
      "hash_key": "083263112afe6caa",
      "page_no": 51,
      "my_file_page_content_table_name": "algolab-askyourpdf-dev-my_file_page_content"
    }
  ]
]

In [10]:
context = []
lambda_handler(event, context)

{'file_name': 'Global AI Industrial Shift.pdf',
 'username': 'hmkim@gsenergy.co.kr',
 'hash_key': '083263112afe6caa',
 'msg': '파일의 읽기, 지식화, 요약, 페이지별 번역/요약이 완료되었습니다'}